In [1]:
from ultralytics import YOLO
import os
import json
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from tqdm import tqdm

In [2]:
dataset_base_path = './data/belgiumts/sequences_jpg'
coco_file = os.path.join(dataset_base_path, 'seq01_01_label_studio_out_coco.json')

In [3]:
model = YOLO('kursova.pt')

In [4]:
dt = []

with open(coco_file, 'r') as f:
    coco_gt = json.load(f)

for img in tqdm(coco_gt['images']):
    img_path = os.path.join(dataset_base_path, img['file_name'])
    # preds = model.predict(img_path, agnostic_nms=False, verbose=False)
    preds = model.track(img_path, tracker='bytetrack.yaml', agnostic_nms=False, verbose=False)
    for (cls, conf, xyxy) in zip(preds[0].boxes.cls, preds[0].boxes.conf, preds[0].boxes.xyxy):
        area_coef = (640.0 / max(preds[0].orig_shape))**2
        x = xyxy[0].item()
        y = xyxy[1].item()
        w = xyxy[2].item() - x
        h = xyxy[3].item() - y
        dt.append({
            'image_id': img['id'],
            'category_id': int(cls.item()),
            'bbox': [x, y, w, h],
            'area': w * h * area_coef,
            'score': conf.item(),
            'iscrowd': 0
        })

with open('./out/seq01_01_dt_coco_track.json', 'w') as f:
    json.dump(dt, f)

100%|██████████| 3001/3001 [01:03<00:00, 47.34it/s]


In [5]:
coco_gt = COCO(coco_file)
coco_dt = coco_gt.loadRes('./out/seq01_01_dt_coco_track.json')
coco_eval = COCOeval(coco_gt, coco_dt, 'bbox')
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.15s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=3.33s).
Accumulating evaluation results...
DONE (t=0.48s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.426
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.475
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.460
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.147
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.482
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.859
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.468
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.475
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets

In [6]:
coco_gt = COCO(coco_file)
coco_dt = coco_gt.loadRes('./out/seq01_01_dt_coco.json')
coco_eval = COCOeval(coco_gt, coco_dt, 'bbox')
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=3.45s).
Accumulating evaluation results...
DONE (t=0.44s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.399
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.440
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.428
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.133
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.452
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.849
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.436
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.443
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets